# Coronspec Tools example notebook

This notebook demonstrates how to use the coronspec tools library to work with HST-17092 data. It's a work-in-progress that will evolve as the coronspec library is built out.

At the end of this notebook, we will print the inferred positions of the occulted primaries.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
import matplotlib as mpl
from matplotlib import pyplot as plt

In [4]:
mpl.rcParams.update({'image.aspect':  'auto', 'image.origin': 'lower'})

In [5]:
from coronspec_tools import utils as ctutils
from coronspec_tools import misc as ctmisc

## List and organize the data files

I like to use a Pandas dataframe to organize my data files by metadata. Since all data files of the same type (e.g. sx1, sx2, flt, et cetera) have the same header keywords, they fit neatly into a dataframe format where the columns represent the keyword value and each row is a separate file. `coronspec_tools.utils` has some functions for setting this up.

In [11]:
# First, let's list all the data files available. Set your path as appropriate.
data_files = sorted(Path("../../../data/MAST_2025-12-16T2001/HST/").glob("of*/*fits"))

In [12]:
# here are all the files:
for i in data_files:
    print(i)

../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_asn.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_crj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_flt.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_raw.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_sx1.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_sx2.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_wav.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_asn.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_crj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_flt.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_raw.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_sx2.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_wav.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01030/of0i01030_asn.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01030

In [13]:
# for each file type, let's make a separate "file manager" dataframe using utils.organize_files_by_header(list_of_files)
file_managers = {}
for f in data_files:
    ftype = f.stem.split("_")[1]
    if ftype not in file_managers.keys():
        file_managers[ftype] = []
    file_managers[ftype].append(f)
for ft in file_managers:
    file_managers[ft] = ctutils.organize_files_by_header(file_managers[ft])

In [14]:
print("The available filetypes are:", ', '.join(sorted(file_managers.keys())))

The available filetypes are: asn, crj, flt, raw, sx1, sx2, wav


In [15]:
asn_files = file_managers.pop("asn")

## Defringing
See https://stistools.readthedocs.io/en/latest/defringe_guide.html

In [25]:
from coronspec_tools import defringing_tools
import stistools

In [17]:
from astropy.io import fits
fits.info(file_managers['wav'].iloc[0]['filepath'])

Filename: /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_wav.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     224   ()      
  1  SCI           1 ImageHDU       120   (1062, 1044)   int16 (rescales to uint16)   
  2  ERR           1 ImageHDU        62   ()      
  3  DQ            1 ImageHDU        45   ()      
  4  SCI           2 ImageHDU       120   (1062, 1044)   int16 (rescales to uint16)   
  5  ERR           2 ImageHDU        62   ()      
  6  DQ            2 ImageHDU        47   ()      


In [18]:
all_files = pd.concat({k: fm[['OBSET_ID', 'filestem', 'FILENAME', 'filepath', 'TARGNAME', 'FRNGFLAT']] for k, fm in file_managers.items()}).reset_index(names=['file_type', 'file_index'])

In [19]:
obset1 = all_files.query("OBSET_ID == '01'")

In [20]:
obset1

,file_type,file_index,OBSET_ID,filestem,FILENAME,filepath,TARGNAME,FRNGFLAT
0,crj,0,01,of0i01010,of0i01010_crj.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
1,crj,1,01,of0i01020,of0i01020_crj.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
2,crj,2,01,of0i01030,of0i01030_crj.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
10,flt,0,01,of0i01010,of0i01010_flt.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
11,flt,1,01,of0i01020,of0i01020_flt.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
12,flt,2,01,of0i01030,of0i01030_flt.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
13,flt,3,01,of0i01040,of0i01040_flt.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,CCDFLAT,N/A
24,raw,0,01,of0i01010,of0i01010_raw.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
25,raw,1,01,of0i01020,of0i01020_raw.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040
26,raw,2,01,of0i01030,of0i01030_raw.fits,/Users/jaguilar/Projects/Research/hst17092-sti...,TYC-1262-187-1,OF0I01040


In [21]:
sci_stem = 'of0i01030'

In [65]:
sci_file_meta = all_files.query("filestem == @sci_stem and file_type == 'raw'").squeeze()

flat_file = Path(all_files.query(f"filestem == '{sci_file_meta['FRNGFLAT'].lower()}' and file_type == 'raw'").squeeze()['filepath'])
sci_file = Path(sci_file_meta['filepath'])
wavecal_file = Path(all_files.query("filestem == @sci_stem and file_type == 'wav'").squeeze()['filepath'])

sci_file, flat_file, wavecal_file = sci_file.resolve(), flat_file.resolve(), wavecal_file.resolve()

In [23]:
str(flat_file).replace("raw","nsp")

'/Users/jaguilar/Projects/Research/hst17092-stis_coron/data/MAST_2025-12-16T2001/HST/of0i01040/of0i01040_nsp.fits'

In [24]:
stistools.defringe.normspflat(
    str(flat_file),
    str(flat_file.name.replace("raw","nsp")),
    do_cal=True,
    wavecal=f"{sci_file}_wav.fits"
)


File written:  /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/of0i01040_crj.fits


In [ ]:
file_managers['flt'].query("filestem == @filestem")['FRNGFLAT']

In [34]:
import shutil

In [ ]:
local_sci_file = shutil.copy(str(sci_file), sci_file.name)
local_flat_file = shutil.copy(str(flat_file), flat_file.name)
local_wavecal_file = shutil.copy(str(wavecal_file), wavecal_file.name)

In [62]:
! rm output/*

drj_file = defringing_tools.defringe_raw(
    str(local_sci_file),
    str(local_flat_file),
    str(local_wavecal_file),
    output_dir='./output/'
)

File written:  /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/output/of0i01040_crj.fits
    
    *** CALSTIS-0 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    16-Dec-2025 15:44:22 EST
    
    Input    of0i01030_raw.fits
    Outroot  output/of0i01030_raw.fits
    Wavecal  of0i01030_wav.fits
    
    *** CALSTIS-1 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    16-Dec-2025 15:44:22 EST
    Input    of0i01030_raw.fits
    Output   output/of0i01030_blv_tmp.fits
    OBSMODE  ACCUM
    APERTURE 52X0.2F1
    OPT_ELEM G750L
    DETECTOR CCD
    
    Imset 1  Begin 15:44:22 EST
    Epcfile  of0i01bxj_epc.fits
    Warning  EPCTAB `of0i01bxj_epc.fits' not found.
    
    CCDTAB   oref$16j1600do_ccd.fits
    CCDTAB   PEDIGREE=GROUND
    CCDTAB   DESCRIP =Updated amp=D gain=4 atodgain and corresponding readnoise values---
    CCDTAB   DESCRIP =Oct. 1996 Air Calibration
    
    DQICORR  PERFORM
    DQITAB   oref$h1v11475o_bpx.fits
   

In [86]:
def process_row(row):
    filestem = row['filestem']
    sci_file = Path(row['filepath']).resolve()
    flat_file = Path(all_files.query(f"filestem == '{row['FRNGFLAT'].lower()}' and file_type == 'raw'").squeeze()['filepath']).resolve()
    wavecal_file = Path(all_files.query("filestem == @filestem and file_type == 'wav'").squeeze()['filepath']).resolve()
    local_sci_file = shutil.copy(str(sci_file), sci_file.name)
    local_flat_file = shutil.copy(str(flat_file), flat_file.name)
    local_wavecal_file = shutil.copy(str(wavecal_file), wavecal_file.name)
    drj_file = defringing_tools.defringe_raw(
        str(local_sci_file),
        str(local_flat_file),
        str(local_wavecal_file),
        output_dir='./output/'
    )
    return drj_file

In [87]:
! rm output/*
all_files.query("file_type == 'raw' and FRNGFLAT != 'N/A'").apply(process_row, axis=1)

File written:  /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/output/of0i01040_crj.fits
    
    *** CALSTIS-0 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    16-Dec-2025 16:03:17 EST
    
    Input    of0i01010_raw.fits
    Outroot  output/of0i01010_raw.fits
    Wavecal  of0i01010_wav.fits
             HSTIO error 114:  Filename of0i01010_wav.fits EXTNAME  EXTVER 0 CFITSIO status 104
    failed to find or open the following file: (ffopen)of0i01010_wav.fits[0]
    Error opening image array.
             Open failed.
    calstis0 failed.
    
    *** CALSTIS-0 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    16-Dec-2025 16:04:20 EST
    
    Input    of0i01010_raw.fits
    Outroot  output/of0i01010_raw.fits
    Wavecal  of0i01010_wav.fits
    
    *** CALSTIS-1 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    16-Dec-2025 16:04:20 EST
    Input    of0i01010_raw.fits
    Output   output/of0i01010_blv_tmp.fits
    OBSMODE  ACCUM

24    output/of0i01010_drj.fits
25    output/of0i01020_drj.fits
26    output/of0i01030_drj.fits
28    output/of0i02010_drj.fits
29    output/of0i02020_drj.fits
30    output/of0i02030_drj.fits
32    output/of0i03010_drj.fits
33    output/of0i03020_drj.fits
35    output/of0i04010_drj.fits
36    output/of0i04020_drj.fits
dtype: object